# LightGBM + exact-time GraphSAGE — frozen post-sweep reproduction

Uses the validation-selected policy from `lgbm_exact_time_middle_threshold_sweep_results`:
- ML bands: low < 0.30, middle [0.30, 0.96), high >= 0.96.
- Graph thresholds: low collision **0.9973355952927457**, all middle rows **0.7590387248013881**, high collision **0.008663969833834228**.
- Noncollision low/high decisions stay direct. The 0.95 graph threshold is retained only for the legacy comparison.

Expected archived TEST: TP 39,880 / FP 626 / FN 430 / TN 1,208,753; F1 **0.9869332805385073**.
Metrics are recomputed from model predictions. Exact model/data identities and validation/TEST confusion counts must match before a successful result export. This notebook does **not** retune thresholds or train models.

Attach the frozen Host input and the schema-5 exact-time GraphSAGE bundle. Their original discovery paths are in the setup cell. NumPy CPU is the default, matching the archived run. Set `USE_CUDA=True` for optional GPU batching; exact confusion checks still apply because floating-point differences near thresholds can change decisions. `RESUME_FROM` is optional; old validation scores and TRAIN registries can be reused after identity checks, but old conditional TEST decisions cannot. Same-output reruns resume automatically.

The earlier TEST was already exposed: this reproduces an exploratory result, not a new blind confirmation. Result ZIP includes computed metrics, all TEST errors/transitions, policy provenance and the executed runtime. The final cell keeps the original blue confusion-matrix style.


In [ ]:
from pathlib import Path
import json, os, sys, tarfile, shutil

MODEL_NAME = 'lgbm'
USE_CUDA = False  # Matches the archived NumPy reference run. True enables optional CUDA.
INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
OUTPUT = WORK / 'lgbm_post_sweep_reproduction_v1'
# Optional prior output containing a TRAIN registry and validation_shards.
# Old conditional TEST decisions are never reused for the post-sweep policy.
RESUME_FROM = None  # Set to an attached prior output directory to reuse scores.
WORK.mkdir(parents=True, exist_ok=True)

# Kaggle may expose the dataset as extracted files or as a single archive.
bundle = INPUT / 'datasets/ngkimngn/optc-cascade-exact-time-graphsage'
required = ('bundle_manifest.json', 'provenance/graph.sqlite', 'provenance/model_numpy.npz',
            'code/cascade_runtime.py', f'models/{MODEL_NAME}_metrics.json')
def is_exact_time_bundle(path):
    if not all((path / name).is_file() for name in required):
        return False
    metadata = json.loads((path / 'bundle_manifest.json').read_text())
    return (metadata.get('schema') == 5 and metadata.get('experiment') == 'exact_time_graphsage_full_cascade'
            and metadata.get('context_policy', {}).get('timestamp_quantum_us') == 1
            and metadata.get('context_policy', {}).get('checkpoint_training_quantum_us') == 1
            and metadata.get('context_policy', {}).get('checkpoint_retrained') is True)

if not is_exact_time_bundle(bundle):
    extracted = [p.parent for p in INPUT.glob('**/bundle_manifest.json')
                 if is_exact_time_bundle(p.parent)]
    if len(extracted) == 1:
        bundle = extracted[0]
    else:
        archives = sorted(INPUT.glob('**/provenance-cascade-exact-time-graphsage.tar.gz'))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one extracted exact-time bundle or archive; found {len(extracted)} extracted, {len(archives)} archives')
        bundle = WORK / 'cascade_bundle_exact_time_graphsage'
        bundle.mkdir(exist_ok=True)
        with tarfile.open(archives[0], 'r:gz') as tar:
            for member in tar.getmembers():
                target = (bundle / member.name).resolve()
                if not target.is_relative_to(bundle.resolve()) or not member.isfile():
                    raise ValueError(f'Unsafe bundle member: {member.name}')
            tar.extractall(bundle)
if not is_exact_time_bundle(bundle):
    raise RuntimeError('This notebook requires the full exact-time GraphSAGE bundle (schema 5)')
bundle_meta = json.loads((bundle / 'bundle_manifest.json').read_text())
assert bundle_meta['context_policy']['timestamp_quantum_us'] == 1
assert bundle_meta['context_policy']['checkpoint_retrained'] is True
assert bundle_meta['context_policy']['checkpoint_training_quantum_us'] == 1
cache_meta = json.loads((bundle / 'provenance/ego_cache_manifest.json').read_text())
training_config = json.loads((bundle / 'provenance/model_training_config.json').read_text())
if cache_meta['manifest_sha256'] != bundle_meta['context_policy']['ego_cache_manifest_sha256']:
    raise ValueError('GraphSAGE training-cache identity mismatch')
if training_config['cache_manifest_sha256'] != cache_meta['manifest_sha256']:
    raise ValueError('GraphSAGE checkpoint/cache mismatch')
if training_config['time_policy']['quantum_us'] != 1 or cache_meta['time_policy']['quantum_us'] != 1:
    raise ValueError('GraphSAGE checkpoint is not exact-time trained')
if training_config['relations'] != cache_meta['relations'] or training_config['attributes'] != len(cache_meta['attributes']):
    raise ValueError('GraphSAGE checkpoint feature/relation schema mismatch')
if cache_meta['relations'] != ['creates', 'communicates', 'communicates_reverse', 'hosts'] or len(cache_meta['attributes']) != 40:
    raise ValueError('Unexpected GraphSAGE node/edge schema')
sys.path.insert(0, str(bundle / 'code'))

host_root = INPUT / 'datasets/nhatnguyen2101/optc-host-ml-full-input'
manifest = host_root / 'manifest.json'
if not (manifest.is_file() and (host_root / 'assets/vocab.json').is_file()):
    candidates = [p for p in INPUT.glob('**/manifest.json')
                  if (p.parent / 'assets/vocab.json').is_file()]
    if len(candidates) == 1:
        manifest = candidates[0]
    else:
        archives = sorted(INPUT.glob('**/host-input.tar'))
        if len(archives) != 1:
            raise RuntimeError(f'Expected one extracted Host dataset or archive; found {len(candidates)} extracted, {len(archives)} archives')
        host_root = WORK / 'host_shards'
        host_root.mkdir(exist_ok=True)
        with tarfile.open(archives[0], 'r') as tar:
            for member in tar.getmembers():
                target = (host_root / member.name).resolve()
                if not target.is_relative_to(host_root.resolve()) or not member.isfile():
                    raise ValueError(f'Unsafe Host archive member: {member.name}')
            tar.extractall(host_root)
        manifest = host_root / 'manifest.json'

print('Host model:', MODEL_NAME)
print('Cascade bundle:', bundle)
print('Host shards:', manifest)
print('Output:', OUTPUT)



In [ ]:
%%writefile /kaggle/working/frozen_lgbm_runtime.py
"""Frozen post-sweep LightGBM + exact-time GraphSAGE reproduction.

Thresholds were selected in the archived validation sweep; no tuning occurs here.
"""
from collections import Counter, deque
from concurrent.futures import ProcessPoolExecutor
import multiprocessing
import hashlib
import json
import os
from pathlib import Path
import sqlite3

import numpy as np

from cascade_runtime import counts, metrics, safe, sha, verify_bundle, load_host_model
from ego import EgoGraphs
from relational_sage import RelationalSAGE


LOW, HIGH, GRAPH, BASELINE = .30, .96, .95, .94016
REFERENCE_IDENTITY = {'baseline_threshold': 0.94016,
 'bundle_sha256': 'f6c8be7575c0617ba600af061d8093b07fb7fd50742c8c3402005860fcf56906',
 'feature_width': 9048,
 'graph_inference_algorithm': 'dense_disjoint_graph_batch_fp64_v1',
 'graph_model_sha256': 'ecf27fd61d2932096d00a79e5d335d49f11552b27e6e9eb4f5fd6b7f51a2a38f',
 'graph_sha256': '33019fb5332a61070411fee4132f6005bee32912aae621fa2ea3345746416d68',
 'graph_threshold': 0.95,
 'high': 0.96,
 'host_model_sha256': '9c0060ab01c0f16cfcbbd3a43226228264b1011d1d697f4f7fc971f6f044a849',
 'key_method': 'exact sorted uint32 column bytes, little-endian',
 'low': 0.3,
 'manifest_sha256': '421d1855c2ae46f01b531f242aaa265a3a23b992d2969b24d534b1b088367fc5',
 'policy_selection': {'high_fn_budget': 10,
                      'low_fp_budget': 10,
                      'minimum_validation_f1_gain': 0.0001},
 'schema_sha256': 'f3e1331a29ff451de2c97af8c016e1e4694bf91a8b506ca2d04f65b96bfbef8e'}
REFERENCE_MANIFEST_SHA = REFERENCE_IDENTITY["manifest_sha256"]
FROZEN_POLICY = {'accepted': True,
 'high_threshold': 0.008663969833834228,
 'low_threshold': 0.9973355952927457,
 'middle_threshold': 0.7590387248013881,
 'reference_middle_threshold': 0.95,
 'runtime_version': 'frozen_middle_sweep_v1',
 'selected': 'swept_threshold',
 'selection_basis': 'Frozen archived validation sweep; all middle rows use the swept threshold; '
                    'direct noncollision low/high gates unchanged.'}
REFERENCE_CONFUSIONS = {'test': {'legacy': {'fn': 3040, 'fp': 473, 'tn': 1208906, 'tp': 37270},
          'selected': {'fn': 430, 'fp': 626, 'tn': 1208753, 'tp': 39880}},
 'validation': {'legacy': {'fn': 2028, 'fp': 330, 'tn': 805922, 'tp': 24845},
                'selected': {'fn': 294, 'fp': 417, 'tn': 805835, 'tp': 26579}}}
REFERENCE_PROVENANCE = {'experiment': 'lgbm_exact_time_middle_threshold_sweep_results',
 'frozen_policy_sha256': '4b9a2f214508ff171e8f49c9f57a59af9a6053473b45b5bf847ff4076d4fce54',
 'note': 'Expected counts are reproduction assertions only; metrics are recomputed from '
         'predictions. Prior TEST exposure makes this exploratory, not a fresh blind evaluation.',
 'test_metrics_sha256': '76e18e5778e702ce3735140097e36e60dc8457a2ec2bee9357f7c7952fff9d26',
 'validation_threshold_sweep_sha256': 'b7d7b33de3d9023dd3323d1b42c9bd0169dab7e729860060fe262c0d121804ad'}

def atomic_json(path, value):
    path = safe(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('w') as stream:
        json.dump(value, stream, indent=2, sort_keys=True)
        stream.flush(); os.fsync(stream.fileno())
    os.replace(tmp, path)

def key_for_row(x, row):
    """Exact, sorted uint32 feature columns; no lossy hash or identity fields."""
    start, stop = int(x.indptr[row]), int(x.indptr[row + 1])
    cols = x.indices[start:stop]
    if len(cols) > 1 and np.any(cols[:-1] >= cols[1:]):
        raise ValueError('Host columns are not strictly increasing')
    return cols.astype('<u4', copy=False).tobytes()

def get_shard(data, shard):
    from host_shards import decode_shard
    return decode_shard(data.path_for(shard), data.width, shard)

def build_registry(data, sampling_seed=1, neg_per_pos=30,
                   expected_fit_counts=(6046559, 201551)):
    """Full TRAIN primary registry and exact frozen undersample diagnostic."""
    full, fit = {}, {}
    full_occurrences, fit_occurrences = {}, {}
    full_rows = Counter(); fit_rows = Counter()
    rng = np.random.default_rng(sampling_seed)
    train = data.meta['splits']['train']
    keep_p = min(1., neg_per_pos * train['label_1'] / train['label_0'])
    for shard in data.meta['shards']:
        if shard['split'] != 'train':
            continue
        x, y, _ = get_shard(data, shard)
        neg = np.flatnonzero(y == 0)
        selected_neg = np.zeros(len(y), dtype=bool)
        selected_neg[neg] = rng.random(len(neg)) < keep_p
        for row, label in enumerate(y):
            label = int(label)
            key = key_for_row(x, row)
            full[key] = full.get(key, 0) | (1 << label)
            if key not in full_occurrences:
                full_occurrences[key] = [0, 0]
            full_occurrences[key][label] += 1
            full_rows[label] += 1
            if label or selected_neg[row]:
                fit[key] = fit.get(key, 0) | (1 << label)
                if key not in fit_occurrences:
                    fit_occurrences[key] = [0, 0]
                fit_occurrences[key][label] += 1
                fit_rows[label] += 1
    if [full_rows[0], full_rows[1]] != [train['label_0'], train['label_1']]:
        raise ValueError('Full TRAIN does not match manifest')
    if expected_fit_counts is not None and [fit_rows[0], fit_rows[1]] != list(expected_fit_counts):
        raise ValueError('Fit subset does not match frozen LightGBM sampling')
    return full, fit, full_rows, fit_rows, full_occurrences, fit_occurrences

def registry_summary(reg, row_counts, occurrences):
    labels = Counter(reg.values())
    collision_rows = [sum(occurrences[key][label] for key, mask in reg.items() if mask == 3)
                      for label in (0, 1)]
    return dict(rows={'benign': row_counts[0], 'malicious': row_counts[1],
                      'total': row_counts[0] + row_counts[1]},
                unique_vectors=len(reg), benign_only_vectors=labels[1],
                malicious_only_vectors=labels[2], conflicting_vectors=labels[3],
                conflicting_vector_rows={'benign': collision_rows[0],
                                         'malicious': collision_rows[1]})

LOW_FP_BUDGET = 10
HIGH_FN_BUDGET = 10
MIN_F1_GAIN = 0.0001
USE_CUDA = False  # Archived reference run used NumPy CPU; optional CUDA is count-checked.
GRAPH_BATCH_SIZE = 256
GRAPH_BUILD_BATCH_SIZE = 64
try:
    _VISIBLE_CPUS = len(os.sched_getaffinity(0))
except (AttributeError, OSError):
    _VISIBLE_CPUS = os.cpu_count() or 2
GRAPH_CPU_WORKERS = max(1, min(8, _VISIBLE_CPUS - 1))
RELATION_NAMES = ('creates', 'communicates', 'communicates_reverse', 'hosts')
INFERENCE_VERSION = 'dense_disjoint_graph_batch_fp64_v1'


class DenseCudaGraphSAGE:
    """Batch disjoint ego graphs across every visible CUDA GPU."""
    def __init__(self, model, batch_size=GRAPH_BATCH_SIZE):
        self.model = model
        self.batch_size = int(batch_size)
        if not USE_CUDA:
            self.torch = None
            self.network = None
            self.device_ids = []
            return
        try:
            import torch
        except ImportError:
            self.torch = None
            self.network = None
            self.device_ids = []
            return
        if not torch.cuda.is_available():
            self.torch = None
            self.network = None
            self.device_ids = []
            return
        self.torch = torch
        self.device_ids = list(range(torch.cuda.device_count()))
        torch.backends.cuda.matmul.allow_tf32 = False

        class BatchNetwork(torch.nn.Module):
            def __init__(self, params):
                super().__init__()
                for name, value in params.items():
                    self.register_buffer('p_' + name, torch.as_tensor(value, dtype=torch.float64))

            def forward(self, x, adjacency, roots, hosts):
                import torch
                h = x
                for layer in (1, 2):
                    z = h @ getattr(self, f'p_W{layer}_self') + getattr(self, f'p_b{layer}')
                    for rid, relation in enumerate(RELATION_NAMES):
                        neighborhood = torch.bmm(adjacency[:, rid], h)
                        z = z + neighborhood @ getattr(self, f'p_W{layer}_{relation}')
                    h = torch.relu(z)
                row = torch.arange(len(roots), device=x.device)
                readout = torch.cat((h[row, roots], h[row, hosts]), dim=1)
                return torch.sigmoid(readout @ self.p_head + self.p_bias[0])

        network = BatchNetwork(model.params).to('cuda:0')
        if len(self.device_ids) > 1:
            self.network = torch.nn.DataParallel(network, device_ids=self.device_ids, output_device=0)
        else:
            self.network = network
        self.verified = False

    @property
    def enabled(self):
        return self.torch is not None

    def _pack(self, graphs):
        batch = len(graphs)
        max_nodes = max(len(g['x']) for g in graphs)
        dim = self.model.input_dim
        x = np.zeros((batch, max_nodes, dim), dtype=np.float64)
        adjacency = np.zeros((batch, len(RELATION_NAMES), max_nodes, max_nodes), dtype=np.float64)
        roots = np.zeros(batch, dtype=np.int64)
        hosts = np.zeros(batch, dtype=np.int64)
        relation_index = {name: i for i, name in enumerate(RELATION_NAMES)}
        for b, graph in enumerate(graphs):
            n = len(graph['x'])
            x[b, :n] = np.asarray(graph['x'], dtype=np.float64)
            roots[b] = int(graph['root'])
            hosts[b] = int(graph['host_root'])
            for src, dst, relation in graph['edges']:
                if relation not in relation_index or src < 0 or dst < 0 or src >= n or dst >= n:
                    raise ValueError('Invalid edge in GraphSAGE batch')
                adjacency[b, relation_index[relation], dst, src] = 1.0
            degree = adjacency[b].sum(axis=2, keepdims=True)
            adjacency[b] /= np.maximum(degree, 1.0)
        return x, adjacency, roots, hosts

    def _score_batch(self, graphs):
        torch = self.torch
        x, adjacency, roots, hosts = self._pack(graphs)
        x = torch.as_tensor(x, device='cuda:0')
        adjacency = torch.as_tensor(adjacency, device='cuda:0')
        roots = torch.as_tensor(roots, device='cuda:0')
        hosts = torch.as_tensor(hosts, device='cuda:0')
        probability = self.network(x, adjacency, roots, hosts).detach().cpu().numpy().astype(np.float64, copy=False)
        if not self.verified:
            reference = np.asarray([self.model.predict(g)['probability'] for g in graphs[:min(8, len(graphs))]])
            error = float(np.max(np.abs(probability[:len(reference)] - reference))) if len(reference) else 0.0
            if error > 2e-8:
                raise ValueError(f'CUDA GraphSAGE batch differs from NumPy reference: max_abs_error={error}')
            self.verified = True
            print(f'CUDA batch verified against NumPy; devices={self.device_ids}; max abs error={error:.3g}', flush=True)
        return probability

    def score(self, graphs):
        if not self.enabled:
            return np.asarray([self.model.predict(g)['probability'] for g in graphs], dtype=np.float64)
        scores = []
        for start in range(0, len(graphs), self.batch_size):
            scores.extend(self._score_batch(graphs[start:start + self.batch_size]))
        return np.asarray(scores, dtype=np.float64)


_GRAPH_WORKER_DB = None
_GRAPH_WORKER_EGO = None

def _init_graph_worker(db_path):
    global _GRAPH_WORKER_DB, _GRAPH_WORKER_EGO
    db_uri = Path(db_path).resolve().as_uri() + '?mode=ro'
    _GRAPH_WORKER_DB = sqlite3.connect(db_uri, uri=True)
    _GRAPH_WORKER_DB.execute('PRAGMA cache_size=-65536')
    _GRAPH_WORKER_DB.execute('PRAGMA mmap_size=1073741824')
    _GRAPH_WORKER_EGO = EgoGraphs(_GRAPH_WORKER_DB)

def _build_graph_chunk(items):
    if _GRAPH_WORKER_EGO is None:
        raise RuntimeError('Graph worker was not initialized')
    rows = [int(row) for row, _ in items]
    graphs = [_GRAPH_WORKER_EGO.build(*target) for _, target in items]
    return rows, graphs

def atomic_npz(path, **arrays):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    with tmp.open('wb') as stream:
        np.savez(stream, **arrays)
        stream.flush(); os.fsync(stream.fileno())
    os.replace(tmp, path)


def policy_fingerprint(policy):
    payload = json.dumps(policy, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(payload).hexdigest()


def checkpoint_identity(identity):
    # Scheduling/device fields are excluded for cache portability; full reference counts are checked.
    return {key:value for key,value in identity.items()
            if key not in {'graph_inference_backend','graph_batch_size',
                           'graph_build_batch_size','graph_cpu_workers'}}


def registry_identity_matches(saved, current):
    # Collision keys depend only on the frozen Host rows, labels and schema.
    required = ('manifest_sha256','schema_sha256','feature_width','key_method')
    return all(saved.get(key) == current.get(key) for key in required)


def inference_backend_name():
    if not USE_CUDA:
        return 'numpy_cpu_reference'
    try:
        import torch
        return f'cuda{torch.cuda.device_count()}x_' + INFERENCE_VERSION if torch.cuda.is_available() else 'numpy_cpu_reference'
    except ImportError:
        return 'numpy_cpu_reference'






def predict(ml, graph, collision, policy):
    ml = np.asarray(ml, dtype=np.float64)
    graph = np.asarray(graph, dtype=np.float64)
    collision = np.asarray(collision, dtype=bool)
    if ml.ndim != 1 or graph.shape != ml.shape or collision.shape != ml.shape:
        raise ValueError('Prediction arrays must be matching vectors')
    if not np.all(np.isfinite(ml)) or np.any((ml < 0) | (ml > 1)):
        raise ValueError('Invalid LightGBM probability')
    middle = (ml >= LOW) & (ml < HIGH)
    low = collision & (ml < LOW)
    high = collision & (ml >= HIGH)
    required = middle | low | high
    if not np.all(np.isfinite(graph[required])) or np.any((graph[required] < 0) | (graph[required] > 1)):
        raise ValueError('Required GraphSAGE probability missing or invalid')
    old = np.where(middle, graph >= GRAPH, ml >= HIGH)
    new = old.copy()
    new[middle] = graph[middle] >= policy['middle_threshold']
    new[low] = graph[low] >= policy['low_threshold']
    new[high] = graph[high] >= policy['high_threshold']
    reason = np.where(low, 'collision_low', np.where(high, 'collision_high_veto',
                      np.where(middle, 'swept_middle', 'legacy_direct')))
    return old, new, reason


def assert_reference_counts(split, legacy, selected):
    measured = dict(legacy=legacy['confusion'], selected=selected['confusion'])
    if measured != REFERENCE_CONFUSIONS[split]:
        raise ValueError(f'{split} does not reproduce the archived sweep: {measured}')



def run(manifest_path, bundle, output, *, registry_source=None, volume_root_ssd=None):
    from host_shards import HostShardDataset
    manifest_path, bundle, output = map(lambda p: Path(p).resolve(),
                                         (manifest_path, bundle, output))
    safe(output)
    if output.is_relative_to(bundle) or output.is_relative_to(manifest_path.parent):
        raise ValueError('Output overlaps input')
    output.mkdir(parents=True, exist_ok=True)
    bundle_meta = verify_bundle(bundle)
    if (bundle_meta.get('schema') != 5 or bundle_meta.get('context_policy', {}).get('timestamp_quantum_us') != 1
            or bundle_meta.get('context_policy', {}).get('checkpoint_training_quantum_us') != 1):
        raise ValueError('Full exact-time retrained GraphSAGE bundle required')
    data = HostShardDataset(manifest_path,
        volume_roots={'ssd': volume_root_ssd} if volume_root_ssd else None)
    identity = dict(manifest_sha256=sha(manifest_path), bundle_sha256=sha(bundle / 'bundle_manifest.json'),
                    graph_sha256=bundle_meta['files']['provenance/graph.sqlite']['sha256'],
                    graph_model_sha256=bundle_meta['files']['provenance/model_numpy.npz']['sha256'],
                    host_model_sha256=bundle_meta['files']['models/lgbm_model.txt']['sha256'],
                    schema_sha256=sha(manifest_path.parent / 'assets/vocab.json'),
                    feature_width=data.width, key_method='exact sorted uint32 column bytes, little-endian',
                    graph_inference_backend=inference_backend_name(), graph_batch_size=GRAPH_BATCH_SIZE,
                    graph_build_batch_size=GRAPH_BUILD_BATCH_SIZE, graph_cpu_workers=GRAPH_CPU_WORKERS,
                    low=LOW, high=HIGH, graph_threshold=GRAPH, baseline_threshold=BASELINE,
                    graph_inference_algorithm=INFERENCE_VERSION,
                    policy_selection=dict(low_fp_budget=LOW_FP_BUDGET, high_fn_budget=HIGH_FN_BUDGET,
                                          minimum_validation_f1_gain=MIN_F1_GAIN))
    if identity['manifest_sha256'] != REFERENCE_MANIFEST_SHA:
        raise ValueError('Frozen Host manifest mismatch')
    baseline_meta = json.loads((bundle / 'models/lgbm_metrics.json').read_text())
    if baseline_meta['validation']['threshold'] != BASELINE:
        raise ValueError('Frozen LightGBM threshold mismatch')
    for filename, field in (('vocab.json','vocabulary_sha256'),
                            ('split_manifest.json','split_sha256')):
        if sha(manifest_path.parent / 'assets' / filename) != baseline_meta[field]:
            raise ValueError('Frozen split/feature asset mismatch')
    if checkpoint_identity(identity) != REFERENCE_IDENTITY:
        raise ValueError('Inputs differ from the archived F1=0.986933 sweep run')
    config_path = output / 'config.json'
    if config_path.exists() and checkpoint_identity(json.loads(config_path.read_text())) != checkpoint_identity(identity):
        raise ValueError('Existing output belongs to another model/data identity')
    atomic_json(config_path, identity)
    policy = dict(FROZEN_POLICY)
    frozen_path = output / 'frozen_policy.json'
    frozen_document = dict(identity=checkpoint_identity(identity), policy=policy,
                           provenance=REFERENCE_PROVENANCE)
    if frozen_path.exists() and json.loads(frozen_path.read_text()) != frozen_document:
        raise ValueError('Existing frozen policy differs; use a new output directory')
    atomic_json(frozen_path, frozen_document)
    atomic_json(output / 'source_index.json',
        {str(s['source_id']): s['source']['path'] for s in data.meta['sources']})
    registry_candidate = Path(registry_source).resolve() if registry_source else output
    registry_files = [registry_candidate / name for name in
                      ('collision_registry.jsonl','collision_registry_summary.json')]
    registry_reused = all(path.is_file() for path in registry_files)
    if registry_reused:
        summary = json.loads(registry_files[1].read_text())
        saved_identity = summary.get('identity')
        if not saved_identity or not registry_identity_matches(saved_identity, identity):
            raise ValueError('TRAIN registry manifest/schema/model identity mismatch')
        registry = {}
        with registry_files[0].open() as stream:
            for line in stream:
                item = json.loads(line)
                if item.get('labels') == [0, 1]:
                    key = bytes.fromhex(item['vector_key'])
                    if key in registry:
                        raise ValueError('Duplicate key in TRAIN collision registry')
                    registry[key] = dict(labels=item['labels'],
                                         train_label_counts=item['train_label_counts'])
        print(f'Reused TRAIN collision registry: {len(registry):,} vectors from {registry_candidate}', flush=True)
    else:
        full, fit, full_rows, fit_rows, full_occ, fit_occ = build_registry(data)
        registry = {k: dict(labels=[0,1], train_label_counts=full_occ[k])
                    for k,v in full.items() if v == 3}
        summary = dict(primary=registry_summary(full, full_rows, full_occ),
                       fit_only_diagnostic=registry_summary(fit, fit_rows, fit_occ),
                       identity=identity)
    atomic_json(output / 'collision_registry_summary.json', summary)
    with (output / 'collision_registry.jsonl').open('w') as stream:
        for key,item in sorted(registry.items()):
            stream.write(json.dumps(dict(vector_key=key.hex(), **item))+'\n')
    scorer_fn, version = load_host_model(bundle, 'lgbm')
    if version != bundle_meta['model_identities']['lgbm']['library_version']:
        raise ValueError('LightGBM library version mismatch')
    def host_scores(x):
        return np.asarray(scorer_fn(x), dtype=np.float64)
    scorer = version
    model = None
    db = None
    ego = None
    graph_pool = None
    batch_scorer = None
    def graph_scores(split, index, shard, lines, routed):
        nonlocal model, db, ego, graph_pool, batch_scorer
        if model is None:
            model = RelationalSAGE.load(bundle / 'provenance/model_numpy.npz')
            batch_scorer = DenseCudaGraphSAGE(model)
            backend = f'CUDA devices={batch_scorer.device_ids}' if batch_scorer.enabled else 'NumPy CPU'
            print(f'GraphSAGE inference: {backend}; graph CPU workers={GRAPH_CPU_WORKERS}', flush=True)
            db_path = bundle / 'provenance/graph.sqlite'
            db = sqlite3.connect(db_path.as_uri() + '?mode=ro', uri=True)
            db.execute('PRAGMA cache_size=-262144')
            db.execute('PRAGMA mmap_size=1073741824')
            if GRAPH_CPU_WORKERS > 1:
                graph_pool = ProcessPoolExecutor(max_workers=GRAPH_CPU_WORKERS,
                    mp_context=multiprocessing.get_context('spawn'), initializer=_init_graph_worker,
                    initargs=(str(db_path),))
            else:
                ego = EgoGraphs(db)
        scores = np.full(len(lines), np.nan, dtype=np.float64)
        selected = np.flatnonzero(routed)
        missing = []
        targets = {}
        # Batch target lookups to reduce Python↔SQLite round trips per event.
        for begin in range(0, len(selected), 700):
            rows = selected[begin:begin + 700]
            source_lines = [int(lines[row]) for row in rows]
            placeholders = ','.join('?' for _ in source_lines)
            query = f'SELECT line,host,proc,t FROM target WHERE source=? AND line IN ({placeholders})'
            for line, host, proc, timestamp in db.execute(query, [shard['source_id'], *source_lines]):
                targets[int(line)] = (host, proc, timestamp)
        items = []
        for row in selected:
            target = targets.get(int(lines[row]))
            if target is None or not target[0] or not target[1] or target[2] is None:
                missing.append([shard['source_id'], int(lines[row])])
                continue
            items.append((int(row), target))
        graph_batch, row_batch = [], []
        def score_ready(force=False):
            while len(graph_batch) >= GRAPH_BATCH_SIZE or (force and graph_batch):
                take = min(GRAPH_BATCH_SIZE, len(graph_batch))
                scores[row_batch[:take]] = batch_scorer.score(graph_batch[:take])
                del graph_batch[:take]
                del row_batch[:take]
        if graph_pool is None:
            for row, target in items:
                graph_batch.append(ego.build(*target))
                row_batch.append(row)
                score_ready()
        else:
            chunks = (items[start:start + GRAPH_BUILD_BATCH_SIZE]
                      for start in range(0, len(items), GRAPH_BUILD_BATCH_SIZE))
            pending = deque()
            chunk_count = (len(items) + GRAPH_BUILD_BATCH_SIZE - 1) // GRAPH_BUILD_BATCH_SIZE
            for _ in range(min(2 * GRAPH_CPU_WORKERS, chunk_count)):
                try:
                    pending.append(graph_pool.submit(_build_graph_chunk, next(chunks)))
                except StopIteration:
                    break
            while pending:
                future = pending.popleft()
                chunk_rows, chunk_graphs = future.result()
                row_batch.extend(chunk_rows)
                graph_batch.extend(chunk_graphs)
                score_ready()
                try:
                    pending.append(graph_pool.submit(_build_graph_chunk, next(chunks)))
                except StopIteration:
                    pass
        score_ready(force=True)
        if missing:
            atomic_json(output / f'{split}-missing-{index:04d}.json', missing)
            raise ValueError('Missing required graph targets')
        return scores

    def shard_cache_paths(output, index):
        directory = output / 'test_shards'
        directory.mkdir(parents=True, exist_ok=True)
        payload = directory / f'test-shard-{index:04d}.npz'
        return payload, payload.with_suffix('.receipt.json')

    def save_test_shard(output, index, shard, policy_hash, arrays):
        payload, receipt_path = shard_cache_paths(output, index)
        atomic_npz(payload, **arrays)
        atomic_json(receipt_path, dict(version=2, identity=checkpoint_identity(identity), policy_sha256=policy_hash,
            source_id=shard['source_id'], shard_sha256=shard['sha256'], rows=int(shard['rows']),
            payload_sha256=sha(payload)))

    def load_test_shard(output, index, shard, policy_hash):
        payload, receipt_path = shard_cache_paths(output, index)
        if not payload.exists() or not receipt_path.exists():
            return None
        receipt = json.loads(receipt_path.read_text())
        expected = dict(version=2, identity=checkpoint_identity(identity), policy_sha256=policy_hash,
                        source_id=shard['source_id'], shard_sha256=shard['sha256'], rows=int(shard['rows']))
        if any(receipt.get(k) != v for k, v in expected.items()) or sha(payload) != receipt.get('payload_sha256'):
            raise ValueError(f'TEST shard checkpoint identity/hash mismatch: {index}')
        if sha(data.path_for(shard)) != shard['sha256']:
            raise ValueError(f'Host shard content differs from frozen manifest: {index}')
        with np.load(payload, allow_pickle=False) as file:
            arrays = {name: file[name].copy() for name in file.files}
        required = {'y','lines','ml','graph','collision','old','new'}
        if set(arrays) != required or any(len(value) != int(shard['rows']) for value in arrays.values()):
            raise ValueError(f'Invalid TEST shard checkpoint shape: {index}')
        old, new, _ = predict(arrays['ml'], arrays['graph'], arrays['collision'], policy)
        if (not np.array_equal(old.astype(np.uint8), arrays['old'])
                or not np.array_equal(new.astype(np.uint8), arrays['new'])):
            raise ValueError(f'TEST shard checkpoint predictions do not reproduce: {index}')
        return arrays
    def validation_cache_paths(index):
        directory = output / 'validation_shards'
        directory.mkdir(parents=True, exist_ok=True)
        payload = directory / f'validation-shard-{index:04d}.npz'
        return payload, payload.with_suffix('.receipt.json')

    def save_validation_shard(index, shard, arrays):
        payload, receipt_path = validation_cache_paths(index)
        atomic_npz(payload, **arrays)
        atomic_json(receipt_path, dict(version=2, identity=checkpoint_identity(identity),
            source_id=shard['source_id'], shard_sha256=shard['sha256'], rows=int(shard['rows']),
            collision_rows=int(arrays['collision'].sum()), payload_sha256=sha(payload)))

    def load_validation_shard(index, shard):
        payload, receipt_path = validation_cache_paths(index)
        if not payload.exists() and not receipt_path.exists():
            return None
        if not payload.is_file() or not receipt_path.is_file():
            payload.unlink(missing_ok=True); receipt_path.unlink(missing_ok=True)
            return None
        receipt = json.loads(receipt_path.read_text())
        if receipt.get('version') not in (1, 2):
            raise ValueError('Unsupported validation cache version')
        expected = dict(identity=checkpoint_identity(identity),
                        source_id=shard['source_id'], shard_sha256=shard['sha256'],
                        rows=int(shard['rows']))
        if any(receipt.get(key) != value for key,value in expected.items()):
            raise ValueError(f'Validation shard checkpoint identity mismatch: {index}')
        if sha(payload) != receipt.get('payload_sha256'):
            raise ValueError(f'Validation shard checkpoint checksum mismatch: {index}')
        if sha(data.path_for(shard)) != shard['sha256']:
            raise ValueError(f'Host validation shard differs from frozen manifest: {index}')
        with np.load(payload, allow_pickle=False) as file:
            arrays = {name:file[name].copy() for name in file.files}
        required = {'y','lines','ml','graph','collision','old'}
        expected_arrays = required | ({'key_offsets','key_bytes'} if receipt['version'] == 1 else set())
        if set(arrays) != expected_arrays or any(arrays[name].shape != (int(shard['rows']),) for name in required):
            raise ValueError(f'Invalid validation checkpoint arrays: {index}')
        if int(arrays['collision'].sum()) != receipt.get('collision_rows'):
            raise ValueError('Validation collision count mismatch')
        routed = arrays['collision'] | ((arrays['ml'] >= LOW) & (arrays['ml'] < HIGH))
        if not np.all(np.isfinite(arrays['graph'][routed])):
            raise ValueError(f'Missing required graph scores in validation checkpoint: {index}')
        old = np.where((arrays['ml'] >= LOW) & (arrays['ml'] < HIGH),
                       arrays['graph'] >= GRAPH, arrays['ml'] >= HIGH).astype(np.uint8)
        if not np.array_equal(old, arrays['old']):
            raise ValueError(f'Validation checkpoint predictions do not reproduce: {index}')
        return arrays

    try:
        val_total = np.zeros(4,dtype=np.int64)
        val_selected = np.zeros(4,dtype=np.int64)
        val_transitions = Counter()
        val_count = 0
        for index, shard in enumerate(data.meta['shards']):
            if shard['split'] != 'val':
                continue
            cached = load_validation_shard(index, shard)
            if cached is None:
                x,y,lines = get_shard(data,shard)
                ml = host_scores(x)
                collision = np.fromiter((key_for_row(x,i) in registry for i in range(len(y))),
                                        dtype=bool,count=len(y))
                middle = (ml >= LOW) & (ml < HIGH)
                graph = graph_scores('val',index,shard,lines,collision|middle)
                old = np.where(middle, graph >= GRAPH, ml >= HIGH).astype(np.uint8)
                cached = dict(y=y.astype(np.uint8), lines=lines.astype(np.int64),
                    ml=ml.astype(np.float64), graph=graph.astype(np.float64),
                    collision=collision.astype(bool), old=old)
                save_validation_shard(index,shard,cached)
                status='computed'
            else:
                status='resumed'
            y=cached['y']; lines=cached['lines']; ml=cached['ml']; graph=cached['graph']
            collision=cached['collision']; old=cached['old'].astype(bool)
            val_total += counts(y,old)
            val_count += len(y)
            _, new, _ = predict(ml, graph, collision, policy)
            val_selected += counts(y, new)
            for row in np.flatnonzero(old != new):
                kind = ('FP' if y[row] == 0 else 'FN') + ('_corrected' if new[row] == y[row] else '_introduced')
                val_transitions[kind] += 1
            print(f'validation shard {index}: {len(y)} rows, {int(collision.sum())} collisions [{status}]',flush=True)
        if val_count != data.meta['splits']['val']['rows']:
            raise ValueError('Validation row count mismatch')
        validation_metrics = dict(identity=identity, scorer=scorer, policy=policy,
            legacy=metrics(val_total), selected=metrics(val_selected),
            transition_counts=dict(val_transitions), provenance=REFERENCE_PROVENANCE)
        atomic_json(output / 'validation_metrics.json', validation_metrics)
        assert_reference_counts('validation', validation_metrics['legacy'], validation_metrics['selected'])
        # TEST is opened only after the validation policy has been frozen on disk.
        frozen = json.loads((output / 'frozen_policy.json').read_text())
        if frozen['identity'] != checkpoint_identity(identity) or frozen['policy'] != policy:
            raise ValueError('Frozen validation policy changed')
        test_old=np.zeros(4,dtype=np.int64);test_new=np.zeros(4,dtype=np.int64)
        transitions=Counter(); routing=Counter(); test_count=0
        policy_hash = policy_fingerprint(policy)
        errors_tmp=output/'test_errors_full_scores.jsonl.tmp'
        changes_tmp=output/'event_transition_audit.jsonl.tmp'
        with errors_tmp.open('w') as errors, changes_tmp.open('w') as changes:
            for index,shard in enumerate(data.meta['shards']):
                if shard['split'] != 'test':
                    continue
                cached = load_test_shard(output, index, shard, policy_hash)
                if cached is None:
                    x,y,lines=get_shard(data,shard)
                    ml=host_scores(x)
                    collision=np.fromiter((key_for_row(x,i) in registry for i in range(len(y))),
                                          dtype=bool,count=len(y))
                    middle=(ml>=LOW)&(ml<HIGH)
                    routed=middle | (collision if policy['accepted'] else np.zeros(len(y),dtype=bool))
                    graph=graph_scores('test',index,shard,lines,routed)
                    old,new,reason=predict(ml,graph,collision,policy)
                    if np.any((old != new) & ~collision & ~middle):
                        raise ValueError('Direct noncollision prediction changed')
                    cached=dict(y=y.astype(np.uint8), lines=lines.astype(np.int64),
                                ml=ml.astype(np.float64), graph=graph.astype(np.float64),
                                collision=collision.astype(bool), old=old.astype(np.uint8), new=new.astype(np.uint8))
                    save_test_shard(output,index,shard,policy_hash,cached)
                    status='computed'
                else:
                    y=cached['y']; lines=cached['lines']; ml=cached['ml']; graph=cached['graph']
                    collision=cached['collision']; old=cached['old'].astype(bool); new=cached['new'].astype(bool)
                    old,new,reason=predict(ml,graph,collision,policy)
                    status='resumed'
                test_old+=counts(y,old);test_new+=counts(y,new)
                test_count+=len(y)
                middle=(ml>=LOW)&(ml<HIGH)
                routed=middle | (collision if policy['accepted'] else np.zeros(len(y),dtype=bool))
                routing['legacy_graph']+=int(middle.sum())
                routing['selected_graph']+=int(routed.sum())
                routing['newly_routed']+=int((routed&~middle).sum())
                for row in np.flatnonzero(new!=y):
                    errors.write(json.dumps(dict(source_id=shard['source_id'],source_line=int(lines[row]),
                        label=int(y[row]),ml_score=float(ml[row]),graph_score=float(graph[row])
                        if np.isfinite(graph[row]) else None,collision=bool(collision[row]),
                        route_reason=str(reason[row]),prediction=int(new[row]),
                        legacy_prediction=int(old[row]),error='FP' if y[row]==0 else 'FN'))+'\n')
                for row in np.flatnonzero(old!=new):
                    kind=('FP' if y[row]==0 else 'FN')+('_corrected' if new[row]==y[row] else '_introduced')
                    transitions[kind]+=1
                    changes.write(json.dumps(dict(source_id=shard['source_id'],source_line=int(lines[row]),
                        label=int(y[row]),ml_score=float(ml[row]),graph_score=float(graph[row]),
                        collision=bool(collision[row]),old_prediction=int(old[row]),new_prediction=int(new[row]),
                        reason=str(reason[row]),transition=kind))+'\n')
                errors.flush(); changes.flush()
                os.fsync(errors.fileno()); os.fsync(changes.fileno())
                print(f'TEST shard {index}: {len(y)} rows, {int(np.sum(old!=new))} changes [{status}]',flush=True)
        if test_count!=data.meta['splits']['test']['rows']:
            raise ValueError('TEST row count mismatch')
        old_c=metrics(test_old)['confusion'];new_c=metrics(test_new)['confusion']
        reconciled=dict(tp=old_c['tp']+transitions['FN_corrected']-transitions['FN_introduced'],
            fp=old_c['fp']-transitions['FP_corrected']+transitions['FP_introduced'],
            fn=old_c['fn']-transitions['FN_corrected']+transitions['FN_introduced'],
            tn=old_c['tn']+transitions['FP_corrected']-transitions['FP_introduced'])
        if new_c!=reconciled:
            raise ValueError('TEST transitions do not reconcile')
        os.replace(errors_tmp, output / 'test_errors_full_scores.jsonl')
        os.replace(changes_tmp, output / 'event_transition_audit.jsonl')
        atomic_json(output/'test_metrics.json',dict(identity=identity,policy=policy,
            legacy=metrics(test_old),selected=metrics(test_new),routing=dict(routing),
            transition_counts=dict(transitions)))
        assert_reference_counts('test', metrics(test_old), metrics(test_new))
        atomic_json(output / 'reproduction_check.json', dict(matched=True,
            validation=metrics(val_selected), test=metrics(test_new), provenance=REFERENCE_PROVENANCE))
        lines = ['# Frozen post-sweep LightGBM + exact-time GraphSAGE', '',
            'Uses the archived validation-selected low/middle/high policy without retuning. '
            'All metrics below are recomputed from predictions. Prior TEST exposure makes '
            'this a reproduction of an exploratory result, not a fresh blind evaluation.', '',
            f"Graph thresholds: low={policy['low_threshold']:.17g}, "
            f"middle={policy['middle_threshold']:.17g}, high={policy['high_threshold']:.17g}.", '',
            '| Split | Policy | TP | FP | FN | TN | Precision | Recall | F1 | FPR |',
            '| --- | --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |']
        for name, old, new in [('Validation', metrics(val_total), metrics(val_selected)),
                               ('TEST', metrics(test_old), metrics(test_new))]:
            for tag, value in [('legacy', old), ('selected', new)]:
                c = value['confusion']
                lines.append(f"| {name} | {tag} | {c['tp']} | {c['fp']} | {c['fn']} | {c['tn']} | "
                    f"{value['precision']:.6f} | {value['recall']:.6f} | {value['f1']:.6f} | {value['fpr']:.6f} |")
        lines += ['', f'Validation transitions: {dict(val_transitions)}; TEST transitions: {dict(transitions)}.',
                  f"TEST graph calls: {routing['selected_graph']} (legacy {routing['legacy_graph']}).",
                  'Full confusion counts match the archived validation and TEST references.']
        summary_tmp = output / 'summary.md.tmp'
        summary_tmp.write_text('\n'.join(lines) + '\n')
        os.replace(summary_tmp, output / 'summary.md')
        return dict(validation=metrics(val_selected), test=metrics(test_new), policy=policy)
    finally:
        if graph_pool is not None:
            graph_pool.shutdown(wait=True, cancel_futures=True)
        if db is not None:
            db.close()


In [ ]:
sys.path.insert(0, str(WORK))
import frozen_lgbm_runtime
from frozen_lgbm_runtime import run, FROZEN_POLICY
frozen_lgbm_runtime.USE_CUDA = USE_CUDA

if RESUME_FROM is not None:
    resume_source = Path(RESUME_FROM).resolve()
    if not resume_source.is_dir() or resume_source == OUTPUT.resolve():
        raise ValueError('RESUME_FROM must be a separate existing output directory')
    OUTPUT.mkdir(parents=True, exist_ok=True)
    names = ['collision_registry.jsonl', 'collision_registry_summary.json', 'validation_shards']
    saved_policy = resume_source / 'frozen_policy.json'
    if saved_policy.is_file() and json.loads(saved_policy.read_text()).get('policy') == FROZEN_POLICY:
        names += ['test_shards']
    copied = 0
    for name in names:
        source = resume_source / name
        files = [source] if source.is_file() else sorted(source.rglob('*')) if source.is_dir() else []
        for source_file in files:
            if not source_file.is_file():
                continue
            target = OUTPUT / source_file.relative_to(resume_source)
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                continue  # Keep newer local checkpoints; runtime validates their receipts.
            shutil.copy2(source_file, target)
            copied += 1
    print('Resume files copied:', copied)

volume_root = manifest.parent / 'volumes/ssd'
if not volume_root.is_dir():
    raise FileNotFoundError(f'Missing Host shard volume: {volume_root}')
result = run(manifest, bundle, OUTPUT, volume_root_ssd=volume_root, registry_source=OUTPUT)
print('Frozen policy:', result['policy'])
print('Recomputed TEST F1:', result['test']['f1'])


In [ ]:
import json, zipfile
from collections import Counter
required = ('config.json', 'source_index.json', 'collision_registry_summary.json',
            'collision_registry.jsonl', 'frozen_policy.json', 'validation_metrics.json',
            'reproduction_check.json',
            'test_metrics.json', 'test_errors_full_scores.jsonl',
            'event_transition_audit.jsonl', 'summary.md')
for name in required:
    if not (OUTPUT / name).is_file():
        raise FileNotFoundError(OUTPUT / name)
validation = json.loads((OUTPUT / 'validation_metrics.json').read_text())
test = json.loads((OUTPUT / 'test_metrics.json').read_text())
frozen = json.loads((OUTPUT / 'frozen_policy.json').read_text())
if frozen['policy'] != validation['policy'] or frozen['policy'] != test['policy']:
    raise ValueError('Validation-frozen policy identity changed')
check = json.loads((OUTPUT / 'reproduction_check.json').read_text())
if (not check.get('matched') or check['validation'] != validation['selected']
        or check['test'] != test['selected']):
    raise ValueError('Reproduction check does not match computed metrics')
errors = Counter()
error_keys = set()
with (OUTPUT / 'test_errors_full_scores.jsonl').open() as stream:
    for line in stream:
        row = json.loads(line)
        key = (row['source_id'], row['source_line'])
        if key in error_keys or row['label'] == row['prediction']:
            raise ValueError('Invalid or duplicate TEST error')
        error_keys.add(key)
        errors[row['error']] += 1
conf = test['selected']['confusion']
if errors != Counter({'FP': conf['fp'], 'FN': conf['fn']}):
    raise ValueError('Incomplete TEST FP/FN export')
transitions = Counter()
transition_keys = set()
with (OUTPUT / 'event_transition_audit.jsonl').open() as stream:
    for line in stream:
        row = json.loads(line)
        key = (row['source_id'], row['source_line'])
        if key in transition_keys or row['old_prediction'] == row['new_prediction']:
            raise ValueError('Invalid or duplicate TEST transition')
        transition_keys.add(key)
        transitions[row['transition']] += 1
if dict(transitions) != test['transition_counts']:
    raise ValueError('Transition counts do not reconcile')
archive = WORK / 'lgbm_post_sweep_reproduction_results.zip'
archive_tmp = archive.with_suffix('.zip.tmp')
with zipfile.ZipFile(archive_tmp, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in required:
        z.write(OUTPUT / name, arcname=name)
    z.write(WORK / 'frozen_lgbm_runtime.py', arcname='frozen_lgbm_runtime.py')
with zipfile.ZipFile(archive_tmp) as z:
    if z.testzip() is not None:
        raise ValueError('Corrupt result archive')
os.replace(archive_tmp, archive)
print('Download:', archive)
print('TEST errors:', dict(errors), 'prediction changes:', len(transition_keys))


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
for split in ('validation', 'test'):
    d=json.loads((OUTPUT / f'{split}_metrics.json').read_text())
    old,new=d['legacy'],d['selected']
    print(split.upper(), 'legacy:',old['confusion'],'F1=',round(old['f1'],6))
    print(split.upper(), 'selected:',new['confusion'],'F1=',round(new['f1'],6))
    fig,ax=plt.subplots(1,2,figsize=(10,3.5))
    ax[0].bar(['Legacy','Selected'],[old['f1'],new['f1']]);ax[0].set_title(split+' F1')
    ax[1].bar(['Legacy FP','Selected FP','Legacy FN','Selected FN'],
              [old['confusion']['fp'],new['confusion']['fp'],old['confusion']['fn'],new['confusion']['fn']])
    ax[1].tick_params(axis='x',rotation=30);ax[1].set_title(split+' errors')
    plt.tight_layout();plt.show()
display(Markdown((OUTPUT/'summary.md').read_text()))


In [ ]:
# Plot the test confusion matrix
import matplotlib.pyplot as plt
import numpy as np
import json

selected_test = json.loads((OUTPUT / "test_metrics.json").read_text())["selected"]

try:
    conf_matrix = np.array([[selected_test['confusion']['tn'], selected_test['confusion']['fp']],
                            [selected_test['confusion']['fn'], selected_test['confusion']['tp']]])

    fig, ax = plt.subplots(figsize=(8, 6))

    # We use log1p for the colormap because of the massive imbalance between TN and the rest
    cax = ax.matshow(np.log1p(conf_matrix), cmap='Blues')

    # Add text annotations with commas for readability
    labels = [
        [f'TN = {conf_matrix[0,0]:,}', f'FP = {conf_matrix[0,1]:,}'],
        [f'FN = {conf_matrix[1,0]:,}', f'TP = {conf_matrix[1,1]:,}']
    ]

    for i in range(2):
        for j in range(2):
            ax.text(j, i, labels[i][j], ha='center', va='center', fontsize=14, 
                    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='0.3', alpha=0.9))

    plt.title('Confusion Matrix', pad=20, fontsize=16)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('Actual Label', fontsize=14)

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['BENIGN', 'MALICIOUS'], fontsize=12)
    ax.set_yticklabels(['BENIGN', 'MALICIOUS'], fontsize=12)
    ax.xaxis.set_ticks_position('bottom')

    confusion_matrix_path = OUTPUT / 'confusion_matrix.png'
    plt.savefig(confusion_matrix_path, dpi=300, bbox_inches='tight')
    print(f"Saved to {confusion_matrix_path}")
    plt.show()
except Exception as e:
    print(f"Could not plot confusion matrix: {e}")
